# Voice Metrics

Review voice model artifacts and metrics.

Steps:
- Check voice model files.
- Run system scorecard and audit report.
- Summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
summary = {
    'models': {},
    'scorecard_exit': None,
    'audit_exit': None,
}


def run_optional(cmd: list[str]) -> int:
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, cwd=str(REPO_ROOT), env=env)
    print('Return code:', result.returncode)
    return result.returncode


In [ ]:
# Check voice model artifacts.
model_paths = [
    REPO_ROOT / 'models' / 'voice_emotion.pkl',
    REPO_ROOT / 'models' / 'voice_emotion_nn.pt',
]

for path in model_paths:
    if path.exists():
        summary['models'][path.name] = {
            'size_mb': round(path.stat().st_size / 1024**2, 2),
        }
        print(path.name, 'size MB:', summary['models'][path.name]['size_mb'])
    else:
        print('Missing:', path)


In [ ]:
# Run the system scorecard.
scorecard_script = REPO_ROOT / 'scripts' / 'system_scorecard.py'
if scorecard_script.exists():
    summary['scorecard_exit'] = run_optional([PY, 'scripts/system_scorecard.py'])
else:
    print('Missing:', scorecard_script)


In [ ]:
# Build the audit report.
audit_script = REPO_ROOT / 'scripts' / 'build_audit_report.py'
if audit_script.exists():
    summary['audit_exit'] = run_optional([PY, 'scripts/build_audit_report.py'])
else:
    print('Missing:', audit_script)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'evaluation_voice_metrics_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
